# NFHS-5 Socioeconomic Enrichment

**Goal:** pick the most socioeconomically representative indicators from the NFHS-5 district health table and join them onto the facilities dataset.

**Verified during analysis:** 706 districts, 107 numeric indicators (85 with ≥80% coverage). Correlation pruning at |r| > 0.7 over an 18-indicator socioeconomic pool keeps **15 representative features**, one per cluster, with max pairwise |r| 0.68 among them.

**Join path:** facilities zip → pincode directory (majority district per pincode) → NFHS-5 (normalized state + district names). Verified coverage: 1,519 of 2,517 labeled rows (60%) match an NFHS district. Unmatched rows keep NULL district features — the training notebook imputes them, so no rows are lost.

**Output:** `workspace.default.facilities_enriched` — the facilities table plus the 15 district indicators. Consumed by the *Enriched Capacity GPU Training* notebook.

Runs on any environment (Standard serverless is fine — no GPU needed). Just *Run All*.

In [0]:
import re
import numpy as np
import pandas as pd

CATALOG = "databricks_virtue_foundation_dataset_dais_2026"
SCHEMA = "virtue_foundation_dataset"
TBL_FACILITIES = f"{CATALOG}.{SCHEMA}.facilities"
TBL_NFHS = f"{CATALOG}.{SCHEMA}.nfhs_5_district_health_indicators"
TBL_PINCODE = f"{CATALOG}.{SCHEMA}.india_post_pincode_directory"

# Enriched output lands here. Change if you want it somewhere else.
ENRICHED_TABLE = "workspace.default.facilities_enriched"

def to_num(s):
    m = re.match(r"^\s*-?\d+(\.\d+)?\s*$", str(s))
    return float(m.group(0)) if m else np.nan

def to_na(s):
    return s.astype("string").str.strip().str.lower().where(
        lambda x: ~x.isin({"null", "none", "nan", "n/a", "na", "", "-", "unknown"}), other=pd.NA
    )

def norm(s):
    return (s.astype("string").str.strip().str.lower()
            .str.replace(r"\s+", " ", regex=True)
            .str.replace(r"[^a-z ]", "", regex=True).str.strip())

nfhs_raw = spark.table(TBL_NFHS).toPandas()
num_cols = [c for c in nfhs_raw.columns if c not in ("district_name", "state_ut")]
nfhs_num = pd.DataFrame({c: nfhs_raw[c].map(to_num) for c in num_cols})
print(f"NFHS-5: {len(nfhs_raw)} districts | {len(num_cols)} indicators parsed numeric")

In [0]:
# Candidate socioeconomic pool: education, household amenities, wealth proxies,
# demographics, women's empowerment, nutrition. Survey-count columns excluded.
POOL = [
    "female_population_age_6_years_and_above_ever_schooled_pct",
    "women_age_15_49_who_are_literate_pct",
    "women_age_15_49_with_10_or_more_years_of_schooling_pct",
    "population_below_age_15_years_pct",
    "sex_ratio_total_f_per_1000_m",
    "hh_electricity_pct",
    "hh_improved_water_pct",
    "hh_use_improved_sanitation_pct",
    "households_using_clean_fuel_for_cooking_pct",
    "households_using_iodized_salt_pct",
    "hh_member_covered_health_insurance_pct",
    "w20_24_married_before_age_18_years_pct",
    "w15_24_who_use_menstrual_hygiene_pct",
    "women_age_15_49_years_whose_bmi_bmi_is_underweight_bmi_lt_1_pct",
    "all_w15_49_who_are_anaemic_pct",
    "institutional_birth_5y_pct",
    "births_attended_by_skilled_hp_5y_10_pct",
    "child_u5_who_are_stunted_height_for_age_18_pct",
]

pool = nfhs_num[POOL].dropna(axis=1, thresh=int(0.8 * len(nfhs_num)))
corr = pool.corr(method="spearman").abs()

# Greedy correlation pruning at |r| > 0.7: repeatedly keep the indicator most
# connected to the rest of the pool, drop its highly-correlated partners.
# Result: one representative per cluster — minimal redundancy.
remaining = list(pool.columns)
representatives = []
while remaining:
    best = corr.loc[remaining, remaining].mean().idxmax()
    representatives.append(best)
    remaining = [c for c in remaining if c != best and corr.loc[best, c] <= 0.7]

print(f"{len(representatives)} representative socioeconomic indicators:")
for r in representatives:
    print(f"  {r}")

In [0]:
facilities = spark.table(TBL_FACILITIES).toPandas()
facilities["capacity_num"] = pd.to_numeric(to_na(facilities["capacity"]), errors="coerce")

# Majority (district, state) per pincode from the India Post directory.
pin = spark.table(TBL_PINCODE).select("pincode", "district", "statename").toPandas()
pin["d"], pin["st"] = norm(pin["district"]), norm(pin["statename"])
cnt = pin.groupby(["pincode", "d", "st"]).size().rename("n").reset_index()
pin_lookup = (cnt.sort_values("n", ascending=False)
              .drop_duplicates("pincode").set_index("pincode")[["d", "st"]])

# NFHS join keys: normalized (state, district) pairs.
nfhs_features = pd.concat([
    pd.DataFrame({"d": norm(nfhs_raw["district_name"]), "st": norm(nfhs_raw["state_ut"])}).reset_index(drop=True),
    nfhs_num[representatives].reset_index(drop=True),
], axis=1)

# facilities: zip -> majority district -> NFHS indicators.
zips = pd.to_numeric(to_na(facilities["address_zipOrPostcode"]), errors="coerce")
facilities["join_d"] = zips.map(pin_lookup["d"])
facilities["join_st"] = zips.map(pin_lookup["st"])

enriched = facilities.merge(nfhs_features, how="left",
                            left_on=["join_d", "join_st"], right_on=["d", "st"])
matched = enriched["d"].notna()
labeled = enriched["capacity_num"].notna()
print(f"facilities: {len(enriched)} | matched to an NFHS district: {matched.sum()}")
print(f"labeled rows matched: {(matched & labeled).sum()} of {labeled.sum()}")

In [0]:
# Unmatched facilities keep NULL district indicators — the training notebook
# imputes them (median), so no rows are dropped.
out = enriched.copy()
for c in out.columns:
    if out[c].dtype == "string" or out[c].dtype == object:
        out[c] = out[c].astype(object).where(out[c].notna(), None)
for c in representatives:
    out[c] = pd.to_numeric(out[c], errors="coerce")

spark.createDataFrame(out).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(ENRICHED_TABLE)
print(f"wrote {len(out)} rows to {ENRICHED_TABLE}")
display(spark.table(ENRICHED_TABLE).select("name", "capacity", "d", representatives[:3]).limit(5))